In [ ]:
# ============================================================
# Tracker 02.07 — Monthly Sales Transaction Generator
# Complete Replacement Version
# ============================================================

import pandas as pd
import numpy as np
import random
from pathlib import Path


# ============================================================
# 1. SETUP
# ============================================================

np.random.seed(42)
random.seed(42)

project_path = Path.cwd().parent
data_path = project_path / "data" / "synthetic"

customers_file = data_path / "customers_master.csv"
products_file = data_path / "products_master.csv"
stores_file = data_path / "stores_master.csv"
salespersons_file = data_path / "salespersons_master.csv"
promotions_file = data_path / "promotions_master.csv"

print("Project path:", project_path)
print("Data path:", data_path)


# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

required_files = {
    "Customers": customers_file,
    "Products": products_file,
    "Stores": stores_file,
    "Salespersons": salespersons_file,
    "Promotions": promotions_file
}

for name, file_path in required_files.items():

    if not file_path.exists():
        raise FileNotFoundError(
            f"{name} file not found:\n{file_path}"
        )

    print(f"{name} file found.")


# ============================================================
# 3. LOAD MASTER DATA
# ============================================================

customers_df = pd.read_csv(customers_file)
products_df = pd.read_csv(products_file)
stores_df = pd.read_csv(stores_file)
salespersons_df = pd.read_csv(salespersons_file)
promotions_df = pd.read_csv(promotions_file)

print("\nMaster data loaded successfully.")

print("Customers:", customers_df.shape)
print("Products:", products_df.shape)
print("Stores:", stores_df.shape)
print("Salespersons:", salespersons_df.shape)
print("Promotions:", promotions_df.shape)


# ============================================================
# 4. VALIDATE STORE MASTER
# ============================================================

if "store_id" not in stores_df.columns:
    raise ValueError(
        "stores_master.csv does not contain store_id."
    )

stores_df["store_id"] = (
    stores_df["store_id"]
    .astype(str)
    .str.strip()
)

stores_df = (
    stores_df
    .drop_duplicates("store_id")
    .sort_values("store_id")
    .reset_index(drop=True)
)

all_store_ids = stores_df["store_id"].to_numpy()

print("\nStore IDs loaded from CURRENT stores_master.csv:")
print("First store:", all_store_ids[0])
print("Last store :", all_store_ids[-1])
print("Distinct stores:", len(all_store_ids))


# ============================================================
# 5. HARD VALIDATION OF STORE-ID FORMAT
# ============================================================
# Project current store master uses:
# STORE001 ... STORE060
#
# This prevents the old STORE000001 format from
# entering the sales files again.

invalid_store_ids = [
    store_id
    for store_id in all_store_ids
    if not (
        isinstance(store_id, str)
        and store_id.startswith("STORE")
        and len(store_id) == 8
        and store_id[5:].isdigit()
    )
]

if invalid_store_ids:
    raise ValueError(
        "Invalid store_id values found in stores_master.csv:\n"
        + "\n".join(invalid_store_ids[:20])
    )

print("Store ID format validation: PASS")
print("Expected format: STORE001 ... STORE999")


# ============================================================
# 6. CONVERT DATES
# ============================================================

customers_df["registration_date"] = pd.to_datetime(
    customers_df["registration_date"]
)

products_df["launch_date"] = pd.to_datetime(
    products_df["launch_date"]
)

stores_df["opening_date"] = pd.to_datetime(
    stores_df["opening_date"]
)

salespersons_df["joining_date"] = pd.to_datetime(
    salespersons_df["joining_date"]
)

promotions_df["start_date"] = pd.to_datetime(
    promotions_df["start_date"]
)

promotions_df["end_date"] = pd.to_datetime(
    promotions_df["end_date"]
)


# ============================================================
# 7. REMOVE DUPLICATE MASTER KEYS
# ============================================================

customers_df = (
    customers_df
    .drop_duplicates("customer_id")
    .reset_index(drop=True)
)

products_df = (
    products_df
    .drop_duplicates("product_id")
    .reset_index(drop=True)
)

salespersons_df = (
    salespersons_df
    .drop_duplicates("salesperson_id")
    .reset_index(drop=True)
)

promotions_df = (
    promotions_df
    .drop_duplicates("promotion_id")
    .reset_index(drop=True)
)


# ============================================================
# 8. MONTHLY SALES-LINE VOLUMES
# ============================================================

monthly_sales_volume = {

    "2024-01": 53200,
    "2024-02": 50400,
    "2024-03": 56000,
    "2024-04": 57100,
    "2024-05": 58800,
    "2024-06": 60500,
    "2024-07": 57700,
    "2024-08": 60500,
    "2024-09": 64400,
    "2024-10": 75600,
    "2024-11": 81200,
    "2024-12": 89600,

    "2025-01": 57500,
    "2025-02": 54400,
    "2025-03": 60500,
    "2025-04": 61700,
    "2025-05": 63500,
    "2025-06": 65300,
    "2025-07": 62300,
    "2025-08": 65300,
    "2025-09": 69600,
    "2025-10": 81600,
    "2025-11": 87700,
    "2025-12": 96800,

    "2026-01": 61700,
    "2026-02": 58500,
    "2026-03": 65000,
    "2026-04": 66300,
    "2026-05": 68200,
    "2026-06": 70200,
    "2026-07": 66900,
    "2026-08": 70200
}

total_expected_lines = sum(
    monthly_sales_volume.values()
)

print(
    "\nExpected total sales lines:",
    total_expected_lines
)


# ============================================================
# 9. PAYMENT METHODS
# ============================================================

store_payment_methods = np.array([
    "UPI",
    "Credit Card",
    "Debit Card",
    "Cash",
    "Net Banking",
    "Wallet"
])

online_payment_methods = np.array([
    "UPI",
    "Credit Card",
    "Debit Card",
    "Net Banking",
    "Wallet"
])


# ============================================================
# 10. ACTIVE STORE COUNT
# ============================================================

def get_active_store_count(month):

    if month < "2024-07":
        return 52

    elif month < "2025-01":
        return 54

    elif month < "2025-07":
        return 56

    elif month < "2026-01":
        return 58

    elif month < "2026-05":
        return 59

    else:
        return 60


# ============================================================
# 11. CUSTOMER ARRAYS
# ============================================================

customer_ids = customers_df[
    "customer_id"
].to_numpy()

customer_registration_dates = (
    customers_df["registration_date"].to_numpy()
)


# ============================================================
# 12. PRODUCT ARRAYS
# ============================================================

product_ids = products_df[
    "product_id"
].to_numpy()

product_categories = products_df[
    "category"
].to_numpy()

product_prices = products_df[
    "unit_price"
].to_numpy()

product_launch_dates = (
    products_df["launch_date"].to_numpy()
)


# ============================================================
# 13. SALESPERSON LOOKUP
# ============================================================

salesperson_lookup = {}

for store_id in all_store_ids:

    store_people = salespersons_df[
        (salespersons_df["store_id"] == store_id)
        &
        (salespersons_df["status"] == "Active")
    ]

    salesperson_lookup[store_id] = (
        store_people["salesperson_id"]
        .dropna()
        .to_numpy()
    )


# ============================================================
# 14. GENERATE SALES
# ============================================================

output_files = []

global_line_number = 1
global_order_number = 1
total_generated = 0


for month, row_count in monthly_sales_volume.items():

    print("\n" + "=" * 65)
    print("Generating:", month)
    print("Required rows:", row_count)

    month_start = pd.Timestamp(
        f"{month}-01"
    )

    month_end = (
        month_start
        + pd.offsets.MonthEnd(1)
    )

    # --------------------------------------------------------
    # ACTIVE STORES
    # --------------------------------------------------------

    active_store_count = get_active_store_count(month)

    active_store_ids = all_store_ids[
        :active_store_count
    ]

    # --------------------------------------------------------
    # ORDER DATE
    # --------------------------------------------------------

    days_in_month = (
        month_end - month_start
    ).days + 1

    random_day_numbers = np.random.randint(
        0,
        days_in_month,
        size=row_count
    )

    order_dates = (
        month_start
        + pd.to_timedelta(
            random_day_numbers,
            unit="D"
        )
    )

    order_date_values = order_dates.to_numpy()

    # --------------------------------------------------------
    # ORDER TIME
    # --------------------------------------------------------

    random_seconds = np.random.randint(
        0,
        86400,
        size=row_count
    )

    order_times = (
        pd.to_timedelta(
            random_seconds,
            unit="s"
        )
    )

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    store_values = np.random.choice(
        active_store_ids,
        size=row_count
    )

    # --------------------------------------------------------
    # CUSTOMER
    # --------------------------------------------------------

    customer_order = np.argsort(
        customer_registration_dates
    )

    sorted_customer_dates = (
        customer_registration_dates[
            customer_order
        ]
    )

    eligible_customer_counts = np.searchsorted(
        sorted_customer_dates,
        order_date_values,
        side="right"
    )

    eligible_customer_counts = np.maximum(
        eligible_customer_counts,
        1
    )

    random_customer_positions = (
        np.random.random(row_count)
        * eligible_customer_counts
    ).astype(int)

    selected_customer_indexes = (
        customer_order[
            random_customer_positions
        ]
    )

    selected_customer_ids = (
        customer_ids[
            selected_customer_indexes
        ]
    )

    anonymous_mask = (
        np.random.random(row_count) < 0.04
    )

    online_mask = (
        np.random.random(row_count) < 0.25
    )

    sales_channel = np.where(
        online_mask,
        "Online",
        "Store"
    )

    selected_customer_ids = (
        selected_customer_ids.astype(object)
    )

    selected_customer_ids[
        anonymous_mask
        & (sales_channel == "Store")
    ] = None

    # --------------------------------------------------------
    # PRODUCT
    # --------------------------------------------------------

    product_order = np.argsort(
        product_launch_dates
    )

    sorted_product_dates = (
        product_launch_dates[
            product_order
        ]
    )

    eligible_product_counts = np.searchsorted(
        sorted_product_dates,
        order_date_values,
        side="right"
    )

    eligible_product_counts = np.maximum(
        eligible_product_counts,
        1
    )

    random_product_positions = (
        np.random.random(row_count)
        * eligible_product_counts
    ).astype(int)

    selected_product_indexes = (
        product_order[
            random_product_positions
        ]
    )

    selected_product_ids = (
        product_ids[
            selected_product_indexes
        ]
    )

    selected_categories = (
        product_categories[
            selected_product_indexes
        ]
    )

    selected_prices = (
        product_prices[
            selected_product_indexes
        ]
    )

    # --------------------------------------------------------
    # QUANTITY
    # --------------------------------------------------------

    quantities = np.random.choice(
        [1, 2, 3, 4],
        size=row_count,
        p=[0.60, 0.25, 0.10, 0.05]
    )

    # --------------------------------------------------------
    # PROMOTIONS
    # --------------------------------------------------------

    promotion_ids_result = np.full(
        row_count,
        None,
        dtype=object
    )

    discount_values = np.zeros(
        row_count,
        dtype=float
    )

    for _, promotion in promotions_df.iterrows():

        promotion_mask = (
            (order_dates >= promotion["start_date"])
            &
            (order_dates <= promotion["end_date"])
        )

        promotion_mask &= (
            (selected_categories ==
             promotion["applicable_category"])
            |
            (promotion["applicable_category"] == "ALL")
        )

        promotion_mask &= (
            (sales_channel ==
             promotion["applicable_channel"])
            |
            (promotion["applicable_channel"] == "ALL")
            |
            (promotion["applicable_channel"] == "Both")
        )

        promotion_mask &= (
            np.random.random(row_count) < 0.30
        )

        promotion_ids_result[
            promotion_mask
        ] = promotion["promotion_id"]

        discount_values[
            promotion_mask
        ] = float(
            promotion["discount_pct"]
        )

    # --------------------------------------------------------
    # SALESPERSON
    # --------------------------------------------------------

    salesperson_values = np.full(
        row_count,
        None,
        dtype=object
    )

    unique_stores = np.unique(
        store_values
    )

    for store_id in unique_stores:

        store_mask = (
            (store_values == store_id)
            &
            (sales_channel == "Store")
        )

        available_salespersons = (
            salesperson_lookup.get(
                store_id,
                np.array([])
            )
        )

        if len(available_salespersons) > 0:

            salesperson_values[
                store_mask
            ] = np.random.choice(
                available_salespersons,
                size=store_mask.sum()
            )

    # --------------------------------------------------------
    # PAYMENT
    # --------------------------------------------------------

    payment_values = np.empty(
        row_count,
        dtype=object
    )

    online_count = (
        sales_channel == "Online"
    ).sum()

    store_count = (
        sales_channel == "Store"
    ).sum()

    payment_values[
        sales_channel == "Online"
    ] = np.random.choice(
        online_payment_methods,
        size=online_count
    )

    payment_values[
        sales_channel == "Store"
    ] = np.random.choice(
        store_payment_methods,
        size=store_count
    )

    # --------------------------------------------------------
    # ORDER STATUS
    # --------------------------------------------------------

    order_status_values = np.random.choice(
        [
            "Completed",
            "Cancelled"
        ],
        size=row_count,
        p=[
            0.96,
            0.04
        ]
    )

    # --------------------------------------------------------
    # ORDER IDs
    # --------------------------------------------------------

    order_ids = np.array([
        f"ORD{month.replace('-', '')}{str(
            global_order_number + i
        ).zfill(9)}"
        for i in range(row_count)
    ])

    global_order_number += row_count

    # --------------------------------------------------------
    # SALES LINE IDs
    # --------------------------------------------------------

    line_numbers = np.arange(
        global_line_number,
        global_line_number + row_count
    )

    sales_line_ids = np.array([
        f"SL{month.replace('-', '')}{str(
            n
        ).zfill(10)}"
        for n in line_numbers
    ])

    global_line_number += row_count

    # --------------------------------------------------------
    # CREATE DATAFRAME
    # --------------------------------------------------------

    monthly_sales_df = pd.DataFrame({

        "sales_line_id":
            sales_line_ids,

        "order_id":
            order_ids,

        "order_date":
            order_dates.date,

        "order_time":
            order_times,

        "customer_id":
            selected_customer_ids,

        "product_id":
            selected_product_ids,

        "store_id":
            store_values,

        "salesperson_id":
            salesperson_values,

        "promotion_id":
            promotion_ids_result,

        "sales_channel":
            sales_channel,

        "quantity":
            quantities,

        "unit_selling_price":
            np.round(
                selected_prices,
                2
            ),

        "discount_pct":
            np.round(
                discount_values,
                2
            ),

        "payment_method":
            payment_values,

        "order_status":
            order_status_values
    })

    # ========================================================
    # 15. VALIDATION
    # ========================================================

    duplicate_line_ids = (
        monthly_sales_df["sales_line_id"]
        .duplicated()
        .sum()
    )

    invalid_store_ids = sorted(
        set(
            monthly_sales_df["store_id"]
        )
        -
        set(all_store_ids)
    )

    invalid_dates = (
        (
            pd.to_datetime(
                monthly_sales_df["order_date"]
            ) < month_start
        )
        |
        (
            pd.to_datetime(
                monthly_sales_df["order_date"]
            ) > month_end
        )
    ).sum()

    invalid_quantity = (
        monthly_sales_df["quantity"] <= 0
    ).sum()

    invalid_price = (
        monthly_sales_df["unit_selling_price"] <= 0
    ).sum()

    invalid_discount = (
        (monthly_sales_df["discount_pct"] < 0)
        |
        (monthly_sales_df["discount_pct"] > 100)
    ).sum()

    print("Rows generated:", len(monthly_sales_df))
    print("Duplicate sales_line_id:", duplicate_line_ids)
    print("Invalid store IDs:", len(invalid_store_ids))
    print("Invalid dates:", invalid_dates)
    print("Invalid quantities:", invalid_quantity)
    print("Invalid prices:", invalid_price)
    print("Invalid discounts:", invalid_discount)

    if invalid_store_ids:
        raise ValueError(
            "Generated sales contain store IDs not present "
            "in the current stores_master.csv:\n"
            + "\n".join(invalid_store_ids[:20])
        )

    if duplicate_line_ids != 0:
        raise ValueError(
            "Duplicate sales_line_id detected."
        )

    if invalid_dates != 0:
        raise ValueError(
            "Invalid order dates detected."
        )

    if invalid_quantity != 0:
        raise ValueError(
            "Invalid quantity detected."
        )

    if invalid_price != 0:
        raise ValueError(
            "Invalid selling price detected."
        )

    if invalid_discount != 0:
        raise ValueError(
            "Invalid discount detected."
        )

    # ========================================================
    # 16. SAVE MONTHLY FILE
    # ========================================================

    output_file = (
        data_path /
        f"sales_{month}.csv"
    )

    monthly_sales_df.to_csv(
        output_file,
        index=False
    )

    output_files.append(
        output_file
    )

    total_generated += len(
        monthly_sales_df
    )

    print(
        "Saved:",
        output_file.name
    )


# ============================================================
# 17. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("02.07 SALES TRANSACTION GENERATION COMPLETED")
print("=" * 65)

print(
    "Expected sales lines:",
    total_expected_lines
)

print(
    "Generated sales lines:",
    total_generated
)

print(
    "Monthly files:",
    len(output_files)
)

if total_generated != total_expected_lines:
    raise ValueError(
        "Generated total does not match expected total."
    )


# ============================================================
# 18. FINAL STORE-ID VALIDATION
# ============================================================

all_generated_store_ids = set()

for file_path in output_files:

    df_check = pd.read_csv(
        file_path,
        usecols=["store_id"]
    )

    all_generated_store_ids.update(
        df_check["store_id"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

invalid_generated_store_ids = (
    all_generated_store_ids
    -
    set(all_store_ids)
)

print(
    "\nGenerated distinct store IDs:",
    len(all_generated_store_ids)
)

print(
    "Current master distinct store IDs:",
    len(all_store_ids)
)

if invalid_generated_store_ids:

    print(
        "INVALID generated store IDs:",
        sorted(invalid_generated_store_ids)
    )

    raise ValueError(
        "Sales generation failed store-ID validation."
    )

print(
    "Final store-ID validation: PASS"
)


# ============================================================
# 19. CHECK FIRST SALES FILE
# ============================================================

first_file = output_files[0]

first_df = pd.read_csv(
    first_file
)

print(
    "\nFirst file:",
    first_file.name
)

print(
    "Shape:",
    first_df.shape
)

print(
    "Columns:",
    list(first_df.columns)
)

print("\nFirst 5 records:")

display(
    first_df.head()
)

print(
    "\n02.07 completed successfully."
)

print(
    "Sales files now use the CURRENT stores_master.csv store IDs."
)

Project path: c:\Projects\retail_sales_analytics
Data path: c:\Projects\retail_sales_analytics\data\synthetic
Customers file found.
Products file found.
Stores file found.
Salespersons file found.
Promotions file found.

Master data loaded successfully.
Customers: (10000, 14)
Products: (2500, 16)
Stores: (60, 11)
Salespersons: (360, 6)
Promotions: (72, 10)

Store IDs loaded from CURRENT stores_master.csv:
First store: STORE001
Last store : STORE060
Distinct stores: 60
Store ID format validation: PASS
Expected format: STORE001 ... STORE999

Expected total sales lines: 2118200

Generating: 2024-01
Required rows: 53200
Rows generated: 53200
Duplicate sales_line_id: 0
Invalid store IDs: 0
Invalid dates: 0
Invalid quantities: 0
Invalid prices: 0
Invalid discounts: 0
Saved: sales_2024-01.csv

Generating: 2024-02
Required rows: 50400
Rows generated: 50400
Duplicate sales_line_id: 0
Invalid store IDs: 0
Invalid dates: 0
Invalid quantities: 0
Invalid prices: 0
Invalid discounts: 0
Saved: sales_

,sales_line_id,order_id,order_date,order_time,customer_id,product_id,store_id,salesperson_id,promotion_id,sales_channel,quantity,unit_selling_price,discount_pct,payment_method,order_status
0,SL2024010000000001,ORD202401000000001,2024-01-07,0 days 08:36:01,CUST000216,PROD001014,STORE029,EMP000069,NaN,Store,2,52172.63,0.0,Credit Card,Completed
1,SL2024010000000002,ORD202401000000002,2024-01-20,0 days 09:24:04,CUST009747,PROD000514,STORE031,EMP000340,NaN,Store,2,11584.54,0.0,Wallet,Completed
2,SL2024010000000003,ORD202401000000003,2024-01-29,0 days 03:54:48,CUST000396,PROD002450,STORE003,EMP000104,NaN,Store,2,59887.22,0.0,Net Banking,Completed
3,SL2024010000000004,ORD202401000000004,2024-01-15,0 days 02:00:36,CUST002829,PROD000322,STORE013,EMP000282,NaN,Store,1,20574.89,0.0,UPI,Completed
4,SL2024010000000005,ORD202401000000005,2024-01-11,0 days 15:16:07,CUST003593,PROD001687,STORE004,NaN,NaN,Online,1,33432.64,0.0,Credit Card,Completed



02.07 completed successfully.
Sales files now use the CURRENT stores_master.csv store IDs.


: 